# Notebook 5 — CMF Budget-Shared Two-Stage Unlearning

**Paper:** *An Illusion of Unlearning?* (Gao et al., AISTATS 2026 · arXiv:2604.08271v1)

**Key difference from NB4:** SAME total epoch budget (50 epochs), SHARED between Stage 1 and Stage 2.  
NB4's 4b ADDS epochs on top of a complete 50-epoch cmf_static run.  
NB5 REALLOCATES: the last `k_shared` epochs of Phase 1 become Phase 2.

**Algorithm:**
- STAGE 1 — epochs 1 to (50 - k_shared): cmf_static's exact per-epoch loop  
  (recompute_cmf → freeze W → update encoder), for (50 - k_shared) epochs.
- STAGE 2 — epochs (50 - k_shared + 1) to 50: freeze encoder permanently,  
  promote W to trainable (clean CMFWeightsTrainable), run k_shared full epochs  
  of real gradient descent on W (CE loss on retain_loader or retain+forget).

**Research question:**  
"If we sacrifice k epochs of encoder training for k epochs of W training,  
within the SAME total budget, is that a net win?"

**Outputs per configuration:**  
- `{method}_cmf_budgetshared_k{k}_stage1end_{mean_source}_ratio{r}_seed{s}.pt`  
- `{method}_cmf_budgetshared_k{k}_final_{mean_source}_ratio{r}_seed{s}.pt`  

**Combined table (final cell):** NB4 4a, NB4 4b, NB5 stage1end, NB5 final —  
for base_method=scrub, so all 4 conditions are visible side-by-side.

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

In [ ]:
import os, sys, json, random, math, time, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
result = subprocess.run(['git','-C',REPO_DIR,'rev-parse','HEAD'],
                        capture_output=True, text=True)
REPO_COMMIT = result.stdout.strip() or 'main'
print('Repo commit:', REPO_COMMIT)

In [ ]:
CKPT_DATASET_DIR = '/kaggle/input/datasets/kiethe/cmf-notebook1'
_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/cmf_benchmark/cmf_benchmark_config.json',
]
config_path = CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p; CKPT_ROOT_NB1 = os.path.dirname(_p); break
assert config_path, f'Cannot find cmf_benchmark_config.json'
with open(config_path) as f: NB1_CFG = json.load(f)

DATASET     = NB1_CFG['dataset']
ARCH        = NB1_CFG['arch']
NUM_CLASSES = NB1_CFG['num_classes']
SEEDS       = NB1_CFG['seeds']
RATIOS      = NB1_CFG['ratios']
TEST_MODE   = NB1_CFG.get('test_mode', False)

# NB5 config
BASE_METHODS   = ['scrub']               # priority: scrub first
MEAN_SOURCES   = ['train', 'retain']
# NB5: SAME total budget as paper's cmf_static. Total = CMF_EPOCHS_BY_METHOD[method].
# Last k_shared epochs reallocated from Stage 1 → Stage 2.
# Table 4: SCRUB+CMF = 3 epochs total. k_shared=1 leaves 2 Stage-1 epochs.
CMF_EPOCHS_BY_METHOD = {
    'random_label':        2 if TEST_MODE else 4,   # Table 4 line 2390
    'salun':               2 if TEST_MODE else 4,   # Table 4 line 2399
    'grad_ascent_descent': 1 if TEST_MODE else 3,   # Table 4 line 2408
    'scrub':               1 if TEST_MODE else 3,   # Table 4 line 2417
    'tarun':               1 if TEST_MODE else 3,   # Table 4 line 2426
}
# TOTAL_EPOCHS = CMF_EPOCHS_BY_METHOD[base_method] — set per-method in loop
K_SHARED       = [1]   # k=1 gives 2 Stage-1 + 1 Stage-2 for SCRUB; extend to [1,2] sweep
PHASE2_DATA    = ['retain_only', 'retain_plus_forget']

# NB4 results dir (for combined table)
CKPT_ROOT_NB4 = '/kaggle/working/checkpoints/cmf'
CKPT_ROOT     = '/kaggle/working/checkpoints/cmf_budgetshared'
os.makedirs(f'{CKPT_ROOT}/stage1end', exist_ok=True)
os.makedirs(f'{CKPT_ROOT}/final',     exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# TOTAL_EPOCHS is per-method (set per-loop from CMF_EPOCHS_BY_METHOD)
print(f'CMF_EPOCHS_BY_METHOD={CMF_EPOCHS_BY_METHOD}  K_SHARED={K_SHARED}  device={device}')

In [ ]:
import torchvision, torchvision.transforms as transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
full_train = torchvision.datasets.CIFAR10('/kaggle/working/data',train=True,download=True,transform=transform_train)
# full_train_eval: eval transform version for probe/NCC feature extraction
# (paper §3.2 / eq.3: features from D_r ∪ D_f, deterministic at eval time)
full_train_eval = torchvision.datasets.CIFAR10('/kaggle/working/data',train=True,download=False,transform=transform_test)
test_set   = torchvision.datasets.CIFAR10('/kaggle/working/data',train=False,download=True,transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_set,batch_size=256,shuffle=False,num_workers=2)
full_train_eval_loader = torch.utils.data.DataLoader(full_train_eval,batch_size=256,shuffle=False,num_workers=2)
print(f'Train: {len(full_train)}  Test: {len(test_set)}')

In [ ]:
import argparse
from unlearn.cmf_weights import ModelModule
from unlearn.cmf_two_stage import CMFWeightsTrainable
from unlearn import unlear_func

# LRs (Table 4 exclusively, PDF lines 2390–2434; arXiv:2604.08271v1)
HPARAM_SOURCE = 'table4'
CMF_LR = {
    'scrub':               5e-3,   # Table 4 line 2417: CIFAR-10 SCRUB+CMF
    'grad_ascent_descent': 1e-4,   # Table 4 line 2408: CIFAR-10 NegGrad++CMF
    'random_label':        2e-3,   # Table 4 line 2390: CIFAR-10 RL+CMF
    'salun':               2e-3,   # Table 4 line 2399: CIFAR-10 SalUn+CMF
    'tarun':               5e-5,   # Table 4 line 2426: CIFAR-10 UNSIR+CMF
}
# Batch sizes: SCRUB uses 64 (Table 4), others 128
CMF_BATCH = {'scrub': 64}
CMF_BATCH_DEFAULT = 128

def make_cmf_args(base_method, lr, epochs, mean_source, forget_idx, retain_idx, seed=0):
    # utils.test() ZeroDivisionError (stratified: all classes in unlearn_class) fixed in utils.py.
    forget_classes = list(set(full_train.targets[i] for i in forget_idx))
    return argparse.Namespace(
        dataset=DATASET, arch=ARCH, num_classes=NUM_CLASSES,
        class_label_names=list(range(NUM_CLASSES)),
        unlearn_method=f'{base_method}_CMF_RemoveFC',
        unlearn_class=forget_classes,   # real forget classes — used by training perturbation logic
        batch_size=128, test_batch_size=256, lr=lr,
        momentum=0.9, weight_decay=5e-4, epochs_or_steps=epochs,
        seed=seed,   # required by random_label, salun (args.seed used for Generator seeds)
        num_retain_samples=len(retain_idx), num_forget_samples=len(forget_idx),
        grad_norm_clip=1.0,
        # SVD: Table 4 line 2375: CIFAR-10 alpha_r=1000, alpha_f=30, samples=900
        SVD_alpha_r=1000, SVD_alpha_f=30,
        SVD_samples=900, SVD_max_patches=10000,
        freeze_except_last=False,   # required by SVD_unlearn -> get_projection_matrix
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=epochs,
        salun_threshold=0.5,
        # UNSIR: Table 4 line 2366/2426: impair_lr = same as main lr
        tarun_impair_lr=lr, tarun_samples_per_class=1000,
        dry_run=TEST_MODE, no_cuda=False, no_mps=True, gamma=0.5,
        data_path='/kaggle/working/data', remove_FC=True,
        CMFClassifier=True, CMF_momentum=0.9, pretrained=False, temperature=1.0,
        prob_batch_size=256, lp_every=0, mean_source=mean_source,
        repo_commit=REPO_COMMIT, test_mode=TEST_MODE,
    )

@torch.no_grad()
def cmf_extract_features(model, loader):
    model.eval()
    feats, labs = [], []
    for x, y in loader:
        x = x.to(device)
        f = model.extract_features(x)
        z = model._preprocess_feats_for_cmf(f)  # A.5 fix
        feats.append(z.cpu()); labs.append(y)
    return torch.cat(feats), torch.cat(labs)

def run_probe_on_features(Xtr, ytr, Xte, yte, n_epochs=50):
    head = nn.Linear(Xtr.size(1), NUM_CLASSES).to(device)
    opt  = optim.SGD(head.parameters(), lr=1e-2, momentum=0.9)
    ldr  = torch.utils.data.DataLoader(
               torch.utils.data.TensorDataset(Xtr, ytr), batch_size=256, shuffle=True)
    for _ in range(n_epochs):
        head.train()
        for bx, by in ldr:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()
    head.eval()
    with torch.no_grad():
        pred = head(Xte.to(device)).argmax(1).cpu()
    return pred, yte

def run_ncc_from_features(Xtr, ytr, Xte, yte):
    """NCC (NC-4): Euclidean nearest-class-centre in raw feature space.
    Matches evaluation/nc.py::ncc_accuracy_from_features() exactly."""
    means = []
    for c in range(NUM_CLASSES):
        m = (ytr == c)
        means.append(Xtr[m].mean(0) if m.any() else torch.zeros(Xtr.size(1)))
    M = torch.stack(means)  # [C, D]
    pred = torch.cdist(Xte.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(dim=1)
    return pred, yte

def acc_split(pred, true, mask):
    return (
        (pred[~mask] == true[~mask]).float().mean().item() * 100,
        (pred[mask]  == true[mask]).float().mean().item() * 100,
    )

def eval_cmf_three_metrics(model, forget_test_mask, full_train_loader, test_loader):
    model.eval()
    with torch.no_grad():
        preds = torch.cat([model(x.to(device)).argmax(1).cpu() for x,_ in test_loader])
    true = torch.tensor(test_set.targets)
    out_ret, out_fgt = acc_split(preds, true, forget_test_mask)

    Xtr, ytr = cmf_extract_features(model, full_train_loader)
    Xte, yte = cmf_extract_features(model, test_loader)
    lp_pred, _  = run_probe_on_features(Xtr, ytr, Xte, yte)
    ncc_pred, _ = run_ncc_from_features(Xtr, ytr, Xte, yte)
    lp_ret, lp_fgt   = acc_split(lp_pred, true, forget_test_mask)
    ncc_ret, ncc_fgt = acc_split(ncc_pred, true, forget_test_mask)
    return {'output_retain_acc':out_ret,'output_forget_acc':out_fgt,
            'probe_retain_acc':lp_ret, 'probe_forget_acc':lp_fgt,
            'ncc_retain_acc':ncc_ret,  'ncc_forget_acc':ncc_fgt}

print('Helpers ready.')

In [ ]:
# ─── NB5: Budget-shared two-stage loop ────────────────────────────────────────
results_nb5 = []

for ratio in RATIOS:
    for seed in SEEDS:
        tag_split = f'ratio{ratio}_seed{seed}'
        if TEST_MODE: tag_split += '_testmode'

        fpath = f'{CKPT_ROOT_NB1}/splits/forget_indices_{tag_split}.json'
        rpath = f'{CKPT_ROOT_NB1}/splits/retain_indices_{tag_split}.json'
        with open(fpath) as f: forget_idx = json.load(f)
        with open(rpath) as f: retain_idx = json.load(f)

        theta_o_path = f'{CKPT_ROOT_NB1}/pre_train/theta_o_seed{seed}.pt'
        if TEST_MODE: theta_o_path = theta_o_path.replace('.pt', '_testmode.pt')
        ck_o = torch.load(theta_o_path, map_location=device)
        theta_o_state = ck_o.get('model_state_dict', ck_o)

        forget_classes   = list(set(full_train.targets[i] for i in forget_idx))
        # Stratified test mask: same ratio-per-class as NB1 split, applied to test set.
        # Class-membership mask is all-True for stratified splits → breaks retain/forget split.
        import random as _rng_mod; import numpy as _np_mod
        _rng = _rng_mod.Random(seed)
        _test_tgts = _np_mod.array(test_set.targets)
        _forget_test = []
        for _c in range(NUM_CLASSES):
            _cls_test = _np_mod.where(_test_tgts == _c)[0].tolist()
            _n = max(1, int(len(_cls_test) * ratio / 100))
            _forget_test.extend(_rng.sample(_cls_test, _n))
        forget_test_mask = torch.zeros(len(test_set), dtype=torch.bool)
        forget_test_mask[_forget_test] = True

        retain_set = torch.utils.data.Subset(full_train, retain_idx)
        forget_set = torch.utils.data.Subset(full_train, forget_idx)
        retain_loader    = torch.utils.data.DataLoader(retain_set, batch_size=128, shuffle=True, num_workers=2)
        forget_loader    = torch.utils.data.DataLoader(forget_set, batch_size=128, shuffle=True, num_workers=2)
        full_train_loader= torch.utils.data.DataLoader(full_train, batch_size=256, shuffle=False, num_workers=2)

        for base_method in BASE_METHODS:
            for mean_source in MEAN_SOURCES:
                for k_shared in K_SHARED:
                    for phase2_data in PHASE2_DATA:
                        total_epochs  = CMF_EPOCHS_BY_METHOD.get(base_method, 3)
                        stage1_epochs = total_epochs - k_shared
                        if stage1_epochs < 1:
                            print(f'  Skipping: k_shared={k_shared} >= total_epochs={total_epochs}')
                            continue
                        tag_base = (f'{base_method}_cmf_budgetshared_k{k_shared}_'
                                    f'{phase2_data}_{mean_source}_ratio{ratio}_seed{seed}')
                        if TEST_MODE: tag_base += '_testmode'

                        ckpt_s1 = f'{CKPT_ROOT}/stage1end/{tag_base}_stage1end.pt'
                        ckpt_s2 = f'{CKPT_ROOT}/final/{tag_base}_final.pt'

                        # Resumable: skip if final checkpoint exists
                        if os.path.exists(ckpt_s2):
                            print(f'[{tag_base}] final exists — skipping.')
                            ck = torch.load(ckpt_s2, map_location=device)
                            results_nb5.append(ck['metrics'])
                            continue

                        print(f'\n[{tag_base}] total={total_epochs} stage1={stage1_epochs} k={k_shared} phase2={phase2_data}')
                        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

                        lr    = CMF_LR.get(base_method, 1e-3)
                        batch = CMF_BATCH.get(base_method, CMF_BATCH_DEFAULT)
                        args  = make_cmf_args(base_method, lr, total_epochs, mean_source,
                                              forget_idx, retain_idx, seed=seed)
                        args.batch_size = batch
                        if base_method == 'scrub':
                            args.scrub_del_bsz  = 64
                            args.scrub_sgda_bsz = 64
                        model = ModelModule(args).to(device)
                        model.encoder.load_state_dict(theta_o_state, strict=False)

                        mean_loader = full_train_loader if mean_source == 'train' else retain_loader
                        model.eval()
                        model.recompute_cmf(mean_loader, device=device)

                        dispatch_key = f'{base_method}_CMF_RemoveFC'
                        fn = unlear_func[dispatch_key]

                        # ── STAGE 1: cmf_static for (TOTAL - k) epochs ──────
                        t0 = time.time()
                        print(f'  Stage 1: {stage1_epochs} epochs of cmf_static')
                        try:
                            model = fn(
                                args=args, model=model, device=device,
                                retain_loader=retain_loader, forget_loader=forget_loader,
                                train_loader=mean_loader, test_loader=test_loader,
                                optimizer=None, epochs=stage1_epochs,
                                test_forget_loader=forget_loader,
                                train_dataset=full_train,   # required by tarun_CMF_unlearn, SVD_unlearn
                                val_index=retain_idx,       # required by tarun_CMF_unlearn, SVD_unlearn
                            )
                        except Exception as e:
                            print(f'  ERROR Stage 1: {e}'); continue

                        # Final recompute after Stage 1
                        model.eval()
                        model.recompute_cmf(mean_loader, device=device)

                        # Save Stage 1 checkpoint
                        s1_metrics = eval_cmf_three_metrics(model, forget_test_mask,
                                                             full_train_eval_loader, test_loader)
                        s1_metrics.update({'stage': 'stage1end', 'method': base_method,
                                           'mean_source': mean_source, 'k_shared': k_shared,
                                           'phase2_data': phase2_data, 'ratio': ratio, 'seed': seed,
                                           'total_epochs': total_epochs, 'stage1_epochs': stage1_epochs})
                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'config': {
                                'base_method': base_method, 'mean_source': mean_source,
                                'total_epochs': total_epochs, 'k_shared': k_shared,
                                'stage1_epochs': stage1_epochs, 'stage': 'stage1end',
                                'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                                'ratio': ratio, 'seed': seed,
                                'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE,
                            },
                            'seed': seed, 'metrics': s1_metrics,
                        }, ckpt_s1)
                        print(f'  [Stage1End] out R={s1_metrics["output_retain_acc"]:.2f}% '
                              f'F={s1_metrics["output_forget_acc"]:.2f}%  Saved {ckpt_s1}')

                        # ── STAGE 2: freeze encoder, gradient-descend W ──────
                        print(f'  Stage 2: {k_shared} epochs of W gradient descent')

                        # Freeze all encoder parameters
                        for p in model.parameters():
                            p.requires_grad_(False)

                        # Promote W (clean — no buffer deletion)
                        tw = CMFWeightsTrainable(model.CMFweights).to(device)
                        tw.promote()

                        # Stage-2 data
                        if phase2_data == 'retain_plus_forget':
                            from torch.utils.data import ConcatDataset
                            s2_ds  = ConcatDataset([retain_set, forget_set])
                            s2_ldr = torch.utils.data.DataLoader(s2_ds, batch_size=128, shuffle=True)
                        else:
                            s2_ldr = retain_loader

                        # Stage-2 W optimizer: use same LR as Stage 1 (CMF_LR[base_method])
                        opt_w  = optim.SGD([tw.W_param], lr=CMF_LR.get(base_method, 1e-3),
                                           momentum=0.9, weight_decay=1e-4)
                        s2_log = []

                        for ep in range(1, k_shared + 1):
                            model.train()
                            ep_loss = n_batches = 0
                            for xb, yb in s2_ldr:
                                xb, yb = xb.to(device), yb.to(device)
                                opt_w.zero_grad()
                                with torch.no_grad():
                                    f = model.extract_features(xb)
                                    z = model._preprocess_feats_for_cmf(f)
                                logits = tw(z, temperature=1.0)
                                loss   = F.cross_entropy(logits, yb)
                                loss.backward()
                                opt_w.step()
                                ep_loss  += loss.item()
                                n_batches += 1
                                if TEST_MODE: break

                            tw.sync_back()
                            model.eval()
                            m = eval_cmf_three_metrics(model, forget_test_mask,
                                                       full_train_eval_loader, test_loader)
                            print(f'  [S2 ep{ep}] out R={m["output_retain_acc"]:.2f}% '
                                  f'F={m["output_forget_acc"]:.2f}%  '
                                  f'probe R={m["probe_retain_acc"]:.2f}% '
                                  f'F={m["probe_forget_acc"]:.2f}%  '
                                  f'ncc R={m["ncc_retain_acc"]:.2f}% '
                                  f'F={m["ncc_forget_acc"]:.2f}%')
                            s2_log.append({'epoch': stage1_epochs + ep, **m})

                        # Final metrics
                        final_metrics = eval_cmf_three_metrics(model, forget_test_mask,
                                                                full_train_eval_loader, test_loader)
                        wall_min = (time.time() - t0) / 60
                        final_metrics.update({
                            'stage': 'final', 'method': base_method,
                            'mean_source': mean_source, 'k_shared': k_shared,
                            'phase2_data': phase2_data, 'ratio': ratio, 'seed': seed,
                            'total_epochs': total_epochs,
                            'stage1_epochs': stage1_epochs,
                            'wall_clock_minutes': wall_min,
                        })

                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'config': {
                                'base_method': base_method, 'mean_source': mean_source,
                                'total_epochs': total_epochs, 'k_shared': k_shared,
                                'stage1_epochs': stage1_epochs,
                                'phase2_data': phase2_data, 'stage': 'final',
                                'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                                'ratio': ratio, 'seed': seed,
                                'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE,
                            },
                            'seed': seed, 'metrics': final_metrics,
                            'stage2_log': s2_log,
                        }, ckpt_s2)
                        print(f'  Saved {ckpt_s2}')
                        results_nb5.append(final_metrics)

df_nb5 = pd.DataFrame(results_nb5)
df_nb5.to_csv(f'{CKPT_ROOT}/results_nb5_budgetshared.csv', index=False)
print('\n=== NB5 budget-shared results saved ===')

In [ ]:
# ─── Combined 4-condition comparison table for base_method=scrub ─────────────
# Conditions:
#   (1) 4a: cmf_static-50ep
#   (2) 4b: cmf_static-50ep + extra W epochs
#   (3) NB5 stage1end: cmf_static-40ep (NB5 Stage 1 end, k=10)
#   (4) NB5 final:     cmf_static-40ep + 10ep-W-shared-budget

print('\n=== Combined 4-condition table: base_method=scrub, mean_source=train ===')
print('(Forget acc compared against oracle retrain forget acc, NOT against 0%)')

METRIC_COLS = ['output_retain_acc','output_forget_acc',
               'probe_retain_acc','probe_forget_acc',
               'ncc_retain_acc','ncc_forget_acc']

def load_csv_if_exists(path):
    try: return pd.read_csv(path)
    except: return pd.DataFrame()

df_4a = load_csv_if_exists(f'{CKPT_ROOT_NB4}/results_4a_cmf_static.csv')
df_4b = load_csv_if_exists(f'{CKPT_ROOT_NB4}/results_4b_cmf_posthoc.csv')

def fmt_mean_std(df, method, mean_src, extra_filters=None):
    m = df[(df.get('method', pd.Series(['']*len(df))).eq(method) if 'method' in df.columns else True)
           & (df.get('mean_source', pd.Series(['']*len(df))).eq(mean_src) if 'mean_source' in df.columns else True)]
    if extra_filters:
        for col, val in extra_filters.items():
            if col in m.columns:
                m = m[m[col] == val]
    if m.empty: return '(no data)'
    parts = []
    for c in METRIC_COLS:
        if c in m.columns:
            v = m[c].dropna()
            parts.append(f'{c}: {v.mean():.1f}±{v.std():.1f}%' if len(v) else f'{c}: N/A')
    return ' | '.join(parts)

for ratio in RATIOS:
    print(f'\n--- Ratio {ratio} ---')
    print(f'4a (cmf_static-50ep):          {fmt_mean_std(df_4a[df_4a["ratio"]==ratio] if "ratio" in df_4a.columns else df_4a, "scrub", "train")}')
    if not df_4b.empty:
        for k_ph in [10]:
            print(f'4b (cmf_static-50ep+k{k_ph}W):    {fmt_mean_std(df_4b[(df_4b.get("ratio",pd.Series())==ratio) & (df_4b.get("k_posthoc",pd.Series())==k_ph)] if "ratio" in df_4b.columns else df_4b, "scrub", "train")}')

    s1end_rows = df_nb5[(df_nb5['method']=='scrub') & (df_nb5['mean_source']=='train')
                         & (df_nb5.get('stage',pd.Series(['']*len(df_nb5)))=='stage1end')
                         & (df_nb5['ratio']==ratio)] if len(df_nb5) and 'ratio' in df_nb5.columns else pd.DataFrame()
    final_rows = df_nb5[(df_nb5['method']=='scrub') & (df_nb5['mean_source']=='train')
                         & (df_nb5.get('stage',pd.Series(['']*len(df_nb5)))=='final')
                         & (df_nb5['ratio']==ratio)] if len(df_nb5) and 'ratio' in df_nb5.columns else pd.DataFrame()

    def fmt_subdf(sdf):
        if sdf.empty: return '(no data)'
        parts = []
        for c in METRIC_COLS:
            if c in sdf.columns:
                v = sdf[c].dropna()
                parts.append(f'{c}: {v.mean():.1f}±{v.std():.1f}%' if len(v) else f'{c}: N/A')
        return ' | '.join(parts)

    k_val = K_SHARED[0] if K_SHARED else 1
    ep1 = CMF_EPOCHS_BY_METHOD.get('scrub', 3) - k_val
    print(f'NB5 stage1end (cmf-{ep1}ep):   {fmt_subdf(s1end_rows)}')
    print(f'NB5 final (cmf-{ep1}ep+{k_val}ep-W):  {fmt_subdf(final_rows)}')

print('\nAll conditions use same Θ_o, same splits, same evaluation protocol.')
print('Forget accuracy convention: compared against oracle retrain forget acc, NOT against 0%.')